<a href="https://colab.research.google.com/github/pulkitn-analytics/glassdoor-salary-prediction/blob/main/AB_capstone06_glassdoor_salary_pred_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Glassdoor Salary Analysis and Prediction


##### **Project Type**    - EDA + Regression
##### **Contribution**    - Individual


# **Project Summary -**

This project explores Glassdoor job listings to understand salary patterns and build regression models for estimating job salaries. The workflow includes data cleaning, exploratory data analysis (EDA), feature engineering, hypothesis testing, and comparison of Linear Regression, Gradient Boosting, and Random Forest models.

Salary ranges are converted into midpoint estimates, and job-related features such as seniority, location, company attributes, and skills mentioned in job descriptions are explored. Model performance is evaluated using MAE, RMSE, and R². The final model is selected based on the results obtained after correcting the preprocessing and feature-selection steps.


# **GitHub Link -**

https://github.com/pulkitn-analytics/glassdoor-salary-prediction

# **Problem Statement**


The objective of this project is to analyse Glassdoor job listings, identify patterns associated with salary estimates, and build regression models to estimate the average salary midpoint for a job listing.

The analysis explores how salary estimates vary across job titles, seniority levels, locations, and company characteristics. The modelling stage compares different regression algorithms using appropriate evaluation metrics. Since the target is calculated from the advertised salary range, the predictions represent estimated salary midpoints rather than verified salaries actually paid to employees.


# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model Selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# Load Dataset
df = pd.read_csv('/content/glassdoor_jobs.csv')

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(8,4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.show()

### What did you know about your dataset?

The dataset contains job-listing information, including job titles, salary estimates, locations, company attributes, and industry details. Initial inspection was performed using the dataset shape, data types, duplicate counts, and missing-value summaries. The dataset was then cleaned and transformed to prepare it for exploratory analysis and salary modelling.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

### Variables Description

The dataset contains both numerical and categorical variables. The `describe()` method summarises numerical columns, while `nunique()` helps identify the number of distinct values in each column. Additional features are created during data wrangling and feature engineering.


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
df.nunique()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Create Seniority Feature
def seniority(title):
    if 'senior' in title.lower():
        return 'Senior'
    elif 'junior' in title.lower() or 'jr' in title.lower():
        return 'Junior'
    else:
        return 'Mid'

df['seniority'] = df['Job Title'].apply(seniority)

# Extract State
df['job_state'] = df['Location'].apply(lambda x: x.split(',')[-1])

# Company Age
df['company_age'] = df['Founded'].apply(lambda x: 2026 - x if x > 0 else None)



# Cleaning salary column
df = df[df['Salary Estimate'] != '-1']  # remove rows where salary is -1
df = df[~df['Salary Estimate'].str.contains('per hour', case=False)]
df = df[~df['Salary Estimate'].str.contains('employer provided', case=False)]

df['Salary Estimate'] = df['Salary Estimate'].str.replace('$', '')
df['Salary Estimate'] = df['Salary Estimate'].str.replace('K', '')
df['Salary Estimate'] = df['Salary Estimate'].str.replace('Employer Provided Salary:', '')
df['Salary Estimate'] = df['Salary Estimate'].str.split('(').str[0]

# split into min and max
df[['min_salary', 'max_salary']] = df['Salary Estimate'].str.split('-', expand=True)

# convert to numbers
df['min_salary'] = df['min_salary'].astype(int)
df['max_salary'] = df['max_salary'].astype(int)

# Create average salary
df['avg_salary'] = (df['min_salary'] + df['max_salary']) / 2


# Remove unnecessary columns
del df['Unnamed: 0']
del df['Competitors']
del df['Salary Estimate']


In [ ]:
df.info()

### What all manipulations have you done and insights you found?

The main data-wrangling steps were:

- Removed listings with invalid salary estimates and excluded hourly or employer-provided salary entries from this analysis.
- Extracted minimum and maximum values from salary ranges and calculated their midpoint as `avg_salary`.
- Created a seniority category from job titles and extracted location information for regional analysis.
- Derived company age from the founding year.
- Created binary indicators for mentions of Python, SQL, Excel, and machine learning in job descriptions.
- Removed columns that were not required for the analysis or modelling.

These steps prepared the data for salary-distribution analysis, hypothesis testing, and regression modelling. The results describe the listings retained after cleaning, not necessarily every job on Glassdoor.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(8,5))
sns.histplot(df['avg_salary'], bins=30, kde=True)
plt.title("Distribution of Average Salary")
plt.xlabel("Average Salary")
plt.ylabel("Frequency")
plt.show()

##### 1. Why did you pick the specific chart?


Histogram helps to understand the distribution of salary and identify whether it is skewed or normally distributed.


##### 2. What is/are the insight(s) found from the chart?

Most salaries are concentrated in a mid-range, with fewer jobs offering extremely high salaries. The distribution appears slightly right-skewed

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


This insight helps job seekers understand the common salary range and set realistic expectations.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
top_jobs = df['Job Title'].value_counts().head(10).index

plt.figure(figsize=(10,6))
sns.boxplot(x='avg_salary', y='Job Title', data=df[df['Job Title'].isin(top_jobs)])
plt.title("Salary by Job Title")
plt.show()

##### 1. Why did you pick the specific chart?

Boxplot shows salary variation across job roles.

##### 2. What is/are the insight(s) found from the chart?


Different roles have significantly different salary ranges. Senior roles tend to have higher median salaries.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


Helps job seekers choose high-paying roles and helps companies benchmark salaries.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.hist(df['avg_salary'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(df['avg_salary'].mean(), color='red', linestyle='--',
            label=f'Mean: ${df["avg_salary"].mean():.1f}k')
plt.axvline(df['avg_salary'].median(), color='green', linestyle='--',
            label=f'Median: ${df["avg_salary"].median():.1f}k')
plt.xlabel('Salary Estimate (k$)')
plt.ylabel('Frequency')
plt.title(' Salary Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

##### 1. Why did you pick the specific chart?

A histogram was selected to show the distribution of Salary Estimates, highlighting the spread and central tendencies for salary prediction analysis.

##### 2. What is/are the insight(s) found from the chart?

The salary-estimate distribution helps show the typical salary range and whether the data is skewed. The mean, median, and presence of extreme values should be interpreted from the chart and the corresponding descriptive statistics.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding the skewed salary distribution can help businesses set competitive salary ranges and assist job seekers in negotiating salaries, potentially improving hiring strategies and satisfaction.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code

plt.figure(figsize=(8,5))
sns.scatterplot(x='Rating', y='avg_salary', data=df)
plt.title("Salary vs Company Rating")
plt.show()

##### 1. Why did you pick the specific chart?


Scatter plot helps visualize the relationship between company ratings and salary.

##### 2. What is/are the insight(s) found from the chart?

There is no strong direct correlation between company rating and salary, though higher-rated companies sometimes offer slightly better pay.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Employees should not rely solely on ratings when evaluating salary expectations. Companies may improve compensation to match their reputation.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
top_ind = df['Industry'].value_counts().head(10).index

plt.figure(figsize=(10,6))
sns.boxplot(x='Industry', y='avg_salary', data=df[df['Industry'].isin(top_ind)])
plt.xticks(rotation=45 , ha='right')
plt.title("Salary by Industry")

plt.show()

##### 1. Why did you pick the specific chart?


To compare salary differences across industries.

##### 2. What is/are the insight(s) found from the chart?


Certain industries (like tech or finance) offer higher salaries compared to others

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


Professionals can choose industries strategically, and companies must stay competitive within their sector.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code

sns.boxplot(x='seniority', y='avg_salary', data=df)
plt.title("Salary by Seniority")
plt.show()

##### 1. Why did you pick the specific chart?

Boxplot shows salary differences across seniority levels.

##### 2. What is/are the insight(s) found from the chart?

The boxplot compares salary distributions across the Junior, Mid, and Senior categories. Differences in the medians and spread indicate whether estimated salaries vary across these groups. The exact pattern should be confirmed from the displayed chart.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This highlights career growth potential and encourages skill development for higher pay.



#### Chart - 07

In [ ]:
# Chart - 7 visualization code

plt.figure(figsize=(10,6))
sns.boxplot(x='Size', y='avg_salary', data=df)
plt.xticks(rotation=45)
plt.title("Salary vs Company Size")
plt.show()

##### 1. Why did you pick the specific chart?


Boxplot helps in comparing salary distributions across different company sizes

##### 2. What is/are the insight(s) found from the chart?

The boxplot compares salary distributions across company-size categories. Differences in median salaries and variability can indicate whether company size is associated with different salary estimates. The chart should be used to confirm the observed pattern.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.


This suggests that job seekers targeting higher salaries may prefer larger organizations. Companies can benchmark their compensation strategies accordingly.

#### Chart - 8 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

##### 1. Why did you pick the specific chart?


Heatmap shows correlation between numerical variables.

##### 2. What is/are the insight(s) found from the chart?


Average salary shows correlation with some features like company age and rating, but not strongly.

#### Chart - 9 - Pair Plot

In [ ]:
# Pair Plot visualization code
sns.pairplot(df[['avg_salary', 'Rating', 'company_age']])
plt.show()

##### 1. Why did you pick the specific chart?


Pairplot helps visualize relationships between multiple variables.

##### 2. What is/are the insight(s) found from the chart?

The pair plot helps examine relationships between average salary, company rating, and company age. These relationships provide an initial view of the data, but the usefulness of these variables for prediction must be assessed through model evaluation.


## ***5. Hypothesis Testing***

### Based on your chart experiments, define hypothetical statements from the dataset and perform the hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on the exploratory data analysis, the following hypotheses are formulated:

1. Salary differs significantly across job seniority levels.
2. Company size has a significant impact on salary.

### Hypothetical Statement - 1: Salary vs Seniority

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): There is no significant difference in salary across different seniority levels.

Alternate Hypothesis (H1): There is a significant difference in salary across different seniority levels.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import f_oneway

junior = df[df['seniority'] == 'Junior']['avg_salary']
mid = df[df['seniority'] == 'Mid']['avg_salary']
senior = df[df['seniority'] == 'Senior']['avg_salary']

f_stat, p_value = f_oneway(junior, mid, senior)

print("P-value:", p_value)

#### Interpretation of Hypothesis Test 1

The ANOVA test compares the mean salary estimates across Junior, Mid, and Senior categories.

At a 5% significance level:
- If the p-value is below 0.05, reject the null hypothesis and conclude that at least one seniority group has a different mean salary estimate.
- If the p-value is 0.05 or above, fail to reject the null hypothesis; the test does not provide sufficient evidence of a difference in group means.

This test does not identify which specific groups differ.


##### Which statistical test have you done to obtain P-Value?

ANOVA (Analysis of Variance) test.

##### Why did you choose the specific statistical test?


ANOVA is used because we are comparing the mean salary across more than two groups (Junior, Mid, Senior).

### Hypothetical Statement - 2: Salary vs Company Size

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): Mean salary estimates do not differ significantly across company-size categories.

Alternative Hypothesis (H1): At least one company-size category has a different mean salary estimate.


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

sizes = df['Size'].unique()
groups = [df[df['Size'] == size]['avg_salary'] for size in sizes]

from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print("P-value:", p_value)

#### Interpretation of Hypothesis Test 2

The ANOVA test compares mean salary estimates across company-size categories.

At a 5% significance level:
- If the p-value is below 0.05, reject the null hypothesis and conclude that at least one company-size category has a different mean salary estimate.
- If the p-value is 0.05 or above, fail to reject the null hypothesis; the test does not provide sufficient evidence of a difference in group means.

A significant result does not by itself prove that company size causes salary differences.


##### Which statistical test have you done to obtain P-Value?

ANOVA Test


##### Why did you choose the specific statistical test?


Because salary is being compared across multiple categories of company size.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# print(df.isnull().sum()) # Null values in Column "Company Age"
df['company_age'] = df['company_age'].fillna(df['company_age'].median())  # Median Imputation
df.isnull().sum()

#### What all missing value imputation techniques have you used and why did you use those techniques?

Missing values were inspected during data exploration. In the modelling features, missing categorical values were filled with "Unknown", while missing numerical values were filled with their respective medians. The company-age feature was also median-imputed during preprocessing.


### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

# Using IQR method
Q1 = df['avg_salary'].quantile(0.25)
Q3 = df['avg_salary'].quantile(0.75)
IQR = Q3 - Q1

df = df[(df['avg_salary'] >= Q1 - 1.5*IQR) & (df['avg_salary'] <= Q3 + 1.5*IQR)]

##### What all outlier treatment techniques have you used and why did you use those techniques?

The IQR method was used to identify extreme salary values. Outlier treatment can reduce the influence of unusually high or low observations, but some extreme salaries may be genuine. Therefore, the effect of removing these observations should be considered when interpreting the results.


### 3. Categorical Encoding

In [ ]:
# Categorical encoding is handled in the Feature Selection section
# using one-hot encoding after extracting skill indicators.

#### What all categorical encoding techniques have you used & why did you use those techniques?

Categorical variables were converted into numerical indicator columns using one-hot encoding with `pandas.get_dummies()`. This represents nominal categories without imposing an artificial numerical order on them.


### 4. Feature Manipulation and Selection


#### 1. Feature Manipulation

In [ ]:
# Extracting Skills from Job Description
df['Job Description'] = df['Job Description'].astype(str)
df['python'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'python' in x else 0)
df['sql'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'sql' in x else 0)
df['excel'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'excel' in x else 0)
df['machine_learning'] = df['Job Description'].str.lower().apply(lambda x: 1 if 'machine learning' in x else 0)



In [ ]:
df.info()


#### 2. Feature Selection

In [ ]:

# Separate the target from the input features
y = df['avg_salary'].copy()

# Remove the target and columns that directly reveal the target
columns_to_drop = [
    'avg_salary',
    'min_salary',
    'max_salary',
    'Salary Estimate',
    'Job Description'
]

X = df.drop(columns=columns_to_drop, errors='ignore').copy()

# Handle missing values in input features
# Fill categorical columns with "Unknown"
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].fillna('Unknown')

# Fill numerical columns with their median
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())

# Convert categorical columns into one-hot encoded columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Missing values in features:", X.isnull().sum().sum())


##### What all feature selection methods have you used  and why?

Feature selection was performed by excluding the target variable, the minimum and maximum salary components used to calculate the target, the original salary-estimate text, and the raw job description. The engineered skill indicators and other available job and company attributes were retained as candidate predictors.


##### Which all features you found important and why?

Job title, seniority, location, company size, industry, and engineered skill indicators were considered candidate predictors because they describe job and employer characteristics. Their actual importance should be assessed using the fitted model rather than assumed in advance.


### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Data transformation included cleaning salary ranges, deriving new features, handling missing values, encoding categorical variables, and scaling the model inputs. These transformations prepared the data for regression modelling.


### 6. Data Scaling

In [ ]:
# Scaling is performed after the train/test split
# to prevent information from the test set leaking into training.

##### Which method have you used to scale you data and why?

StandardScaler was used to standardise the model input features. It was fitted on the training data only and then applied to the test data to avoid leakage from the test set during preprocessing.


### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?


Dimensionality reduction is not required as the number of features is manageable and does not cause high computational complexity

### 8. Data Splitting

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data before fitting the scaler
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Fit the scaler only on training data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


##### What data splitting ratio have you used and why?


An 80-20 split was used, where 80% data is used for training and 20% for testing. This ensures sufficient data for training while maintaining a reliable test set.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Class-imbalance techniques such as oversampling are generally designed for classification problems and are not directly applicable to this regression task. The distribution of the continuous salary target was instead examined during exploratory data analysis.


## ***7. ML Model Implementation***

### ML Model - 1 : Linear Regression

In [ ]:
# ML Model - 1 Implementation
lr = LinearRegression()

# Fit the model
lr.fit(X_train, y_train)

# Predict
y_pred_lr = lr.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
# Evaluation Metrics

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R2 Score:", r2_lr)

In [ ]:
# Visualizing evaluation Metric Score chart

metrics = ['MAE', 'RMSE', 'R2']
values = [mae_lr, rmse_lr, r2_lr]

plt.bar(metrics, values)
plt.title("Linear Regression Performance")
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

No major tuning for Linear Regression

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.


No major improvement observed

### ML Model - 2 : Gradient Boosting

In [ ]:
# ML Model - 2 Implementation
gb = GradientBoostingRegressor()

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)



#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.


Gradient Boosting builds models sequentially to correct errors from previous models

In [ ]:
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print("MAE:", mae_gb)
print("RMSE:", rmse_gb)
print("R2 Score:", r2_gb)

In [ ]:
# Visualization
metrics = ['MAE', 'RMSE', 'R2']
values = [mae_gb, rmse_gb, r2_gb]

plt.bar(metrics, values)
plt.title("Gradient Boosting Performance")
plt.show()

### ML Model - 3 : Random Forest

In [ ]:
# ML Model - 3 Implementation

rf = RandomForestRegressor(random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Random Forest is an ensemble learning method that combines predictions from multiple decision trees. It can capture nonlinear relationships between job and company characteristics and the target variable. Its performance should be compared with Linear Regression and Gradient Boosting using the test-set MAE, RMSE, and R² scores.


In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

In [ ]:
# Visualization
metrics = ['MAE', 'RMSE', 'R2']
values = [mae_rf, rmse_rf, r2_rf]

plt.bar(metrics, values)
plt.title("Random Forest Performance")
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)

print("Best parameters:", grid.best_params_)
print("Best cross-validation R²:", round(grid.best_score_, 3))


### Post Tuning

In [ ]:

# Compare all models on the same test set

results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Gradient Boosting',
        'Random Forest',
        'Tuned Random Forest'
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_gb),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_rf_tuned)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_gb)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned))
    ],
    'R2': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_gb),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_rf_tuned)
    ]
})

print(results.round(3).to_string(index=False))


##### Which hyperparameter optimization technique have you used and why?


GridSearchCV was used to find the best combination of parameters

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The effect of hyperparameter tuning is assessed by comparing the tuned Random Forest with the original Random Forest using the same test set. The model-comparison results determine whether tuning improved performance.


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

MAE, RMSE, and R² were used to evaluate the regression models. MAE represents the average absolute prediction error, RMSE gives greater weight to larger errors, and R² indicates how much of the variation in the target is explained by the model. MAE and RMSE are interpreted in the same units as the salary target.


### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The final model is selected by comparing the test-set MAE, RMSE, and R² values of Linear Regression, Gradient Boosting, Random Forest, and tuned Random Forest. The model with the most suitable overall results is chosen, considering both prediction error and explained variance.


### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Feature importance can help identify which input features contribute most to the fitted Random Forest model. Any reported ranking should be based on feature-importance values calculated from the fitted model. These values indicate predictive usefulness within the model and do not establish that a feature causes salary differences.


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
import joblib

joblib.dump(best_rf, 'salary_prediction_model.pkl')

print("Tuned Random Forest model saved successfully.")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data
loaded_model = joblib.load('salary_prediction_model.pkl')

# Take one sample
sample = X_test[0].reshape(1, -1)
prediction = loaded_model.predict(sample)

print("Predicted Salary:", prediction)

# **Conclusion**



This project analyses Glassdoor job listings to explore salary patterns and build regression models for estimating salary midpoints. The workflow includes data cleaning, exploratory data analysis, feature engineering, hypothesis testing, and comparison of Linear Regression, Gradient Boosting, and Random Forest models.

The target variable is calculated as the midpoint of the listed minimum and maximum salary estimates. To make model evaluation more reliable, the salary components used to calculate the target are excluded from the input features, categorical variables are one-hot encoded, and scaling is fitted on the training data only.

The final model and its performance are determined from the updated test-set evaluation results. The findings should be interpreted as patterns in the available job listings, not proof of causation or a guarantee of actual salaries paid. Further work could include validating the model on newer listings and improving the treatment of job descriptions and salary data.


###*~ Pulkit Narang*